# Rolls-Royce Holdings plc — Share Price & Event-Study Analysis

**Part 4 of the Rolls-Royce independent equity research project** (Excel model → Python analysis → report & presentation).

## Objective
Analyse the share-price behaviour of Rolls-Royce Holdings plc (RR.L) against the FTSE 100 (^FTSE) from January 2021 to 8 October 2026, to understand:

1. Performance — absolute, relative, annual and monthly returns
2. Risk — volatility, rolling volatility, drawdowns, correlation and beta
3. Risk-adjusted return — Sharpe ratio
4. Information events — how the price reacted to results and trading updates, using simple event windows and a **market-model event study** with significance tests

The outputs feed the valuation model (beta cross-check) and the investment thesis (how much of the turnaround the market has already rewarded).

## Data
Adjusted daily closes from Yahoo Finance via `yfinance` (`auto_adjust=True`, so prices include dividends and corporate actions). RR.L is quoted in **pence**. The first successful download is saved to `data/prices.csv` so the notebook can be re-run offline and the results stay reproducible after Yahoo revises its data.

*Educational and portfolio analysis only — not investment advice.*

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

try:
    from scipy import stats
    HAVE_SCIPY = True
except ImportError:  # p-values fall back to a normal approximation
    HAVE_SCIPY = False

plt.rcParams.update({"figure.figsize": (12, 5.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ---- Analysis settings (edit here, not further down) ----
RR, MKT = "RR.L", "^FTSE"
START, END = "2021-01-01", "2026-10-09"   # yfinance 'end' is exclusive -> last day 8 Oct 2026
TRADING_DAYS = 252
RISK_FREE = 0.035      # average UK Bank Rate over 2021-2026 was roughly 3-4%; 0.1% in 2021, 5.25% peak in 2023-24
DATA_FILE = Path("data/prices.csv")
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

## 1. Load and clean the data

In [ ]:
if DATA_FILE.exists():
    prices = pd.read_csv(DATA_FILE, index_col=0, parse_dates=True)
    print(f"Loaded cached data from {DATA_FILE}")
else:
    import yfinance as yf
    raw = yf.download([RR, MKT], start=START, end=END, auto_adjust=True, progress=False)
    prices = raw["Close"][[RR, MKT]].copy()
    DATA_FILE.parent.mkdir(exist_ok=True)
    prices.to_csv(DATA_FILE)
    print(f"Downloaded from Yahoo Finance and cached to {DATA_FILE}")

prices = prices.loc[START:END].sort_index()
missing = prices.isna().sum()
print("Missing values before cleaning:\n", missing.to_string())

# Keep only days on which BOTH series traded, so every return compares like with like
prices = prices.dropna()
print(f"\n{len(prices):,} common trading days: {prices.index[0].date()} to {prices.index[-1].date()}")
prices.tail()

## 2. Price history and relative performance
Both series are rebased to 100 on the first trading day so performance can be compared independently of price level.

In [ ]:
fig, ax = plt.subplots()
ax.plot(prices.index, prices[RR], label="Rolls-Royce (adjusted close)")
ax.set(title="Rolls-Royce Holdings — adjusted share price", ylabel="Pence")
ax.legend(); plt.show()

rebased = prices / prices.iloc[0] * 100
fig, ax = plt.subplots()
ax.plot(rebased.index, rebased[RR], label="Rolls-Royce")
ax.plot(rebased.index, rebased[MKT], label="FTSE 100")
ax.set(title="Rolls-Royce vs FTSE 100 — rebased to 100", ylabel="Index (start = 100)")
ax.legend(); plt.show()

total_return = prices.iloc[-1] / prices.iloc[0] - 1
print("Total return over the period:")
print(total_return.map("{:.1%}".format).to_string())

## 3. Daily returns and annualised volatility
Simple daily return: $R_t = \dfrac{P_t}{P_{t-1}} - 1$. Annualised volatility: $\sigma_{annual} = \sigma_{daily}\sqrt{252}$.

In [ ]:
returns = prices.pct_change().dropna()
ann_vol = returns.std() * np.sqrt(TRADING_DAYS)

display(returns.describe().T)
print("Annualised volatility:")
print(ann_vol.map("{:.1%}".format).to_string())
print(f"\nRolls-Royce volatility is {ann_vol[RR]/ann_vol[MKT]:.1f}x the FTSE 100.")

## 4. Rolling 30-day volatility
Volatility is not constant. A 30-trading-day rolling window shows when risk was concentrated.

In [ ]:
rolling_vol = returns.rolling(30).std() * np.sqrt(TRADING_DAYS)

fig, ax = plt.subplots()
ax.plot(rolling_vol.index, rolling_vol[RR], label="Rolls-Royce")
ax.plot(rolling_vol.index, rolling_vol[MKT], label="FTSE 100")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(title="30-day rolling annualised volatility", ylabel="Annualised volatility")
ax.legend(); plt.show()

peak_date = rolling_vol[RR].idxmax()
print(f"Peak 30-day volatility: {rolling_vol[RR].max():.1%} on {peak_date.date()}")

# Which days drove the spike? Largest absolute moves inside the 30-day window ending on the peak date
window = returns.loc[:peak_date, RR].iloc[-30:]
print(f"\nLargest daily moves in the window {window.index[0].date()} to {window.index[-1].date()}:")
print(window.reindex(window.abs().sort_values(ascending=False).index).head(5).map("{:+.1%}".format).to_string())

### Interpretation
The peak 30-day volatility falls at the end of March 2022, so the window covers late February to March 2022. That period contains Rolls-Royce's FY2021 results (24 February 2022) and Russia's invasion of Ukraine (also 24 February 2022), which hit airlines, aerospace demand and input costs, as well as a sharp sell-off and rebound in March 2022. The table above lists the specific days that drove the spike; check each against the RNS archive before attributing a cause.

Volatility peaks well above the long-run average show why a single full-period volatility figure understates how risky the stock could be at times.

## 5. Correlation and beta
$\beta = \dfrac{\mathrm{Cov}(R_{RR}, R_{M})}{\mathrm{Var}(R_{M})}$. Raw historical betas tend to drift towards 1 over time, so the **Blume-adjusted beta** $= 0.67\,\beta_{raw} + 0.33$ is also shown. A rolling one-year beta shows how stable the estimate is.

In [ ]:
correlation = returns[RR].corr(returns[MKT])
beta = returns[RR].cov(returns[MKT]) / returns[MKT].var()
beta_blume = 0.67 * beta + 0.33

roll_beta = (returns[RR].rolling(TRADING_DAYS).cov(returns[MKT])
             / returns[MKT].rolling(TRADING_DAYS).var())

print(f"Correlation with FTSE 100: {correlation:.3f}")
print(f"Raw beta:                  {beta:.3f}")
print(f"Blume-adjusted beta:       {beta_blume:.3f}")
print(f"Model beta (FT, Excel):    1.582")
print(f"Rolling 1-year beta range: {roll_beta.min():.2f} to {roll_beta.max():.2f}; latest {roll_beta.dropna().iloc[-1]:.2f}")

fig, ax = plt.subplots()
ax.plot(roll_beta.index, roll_beta, label="Rolling 252-day beta")
ax.axhline(beta, linestyle="--", linewidth=1, label=f"Full-period beta {beta:.2f}")
ax.axhline(1, color="grey", linewidth=0.8)
ax.set(title="Rolls-Royce rolling 1-year beta vs FTSE 100", ylabel="Beta")
ax.legend(); plt.show()

### Interpretation
A correlation of about 0.5 means only around a quarter of Rolls-Royce's daily variance is explained by the FTSE 100 (R² ≈ correlation²); most of the risk is company-specific. The full-period beta of about 1.65 supports the 1.58 beta used in the Excel CAPM. The rolling chart and the Blume adjustment show the estimate is not stable. A beta measured through a turnaround may overstate future market sensitivity, which is why the valuation is tested across a range of discount rates.

## 6. Annualised return and Sharpe ratio
$Sharpe = \dfrac{R_p - R_f}{\sigma_p}$, using the geometric annualised return. The risk-free rate is set in the settings cell; a single rate is a simplification because UK rates rose from 0.1% (2021) to 5.25% (2023–24).

In [ ]:
years = len(returns) / TRADING_DAYS
ann_return = (1 + returns).prod() ** (1 / years) - 1
sharpe = (ann_return - RISK_FREE) / ann_vol

risk_table = pd.DataFrame({"Annualised return": ann_return,
                           "Annualised volatility": ann_vol,
                           "Sharpe ratio": sharpe}).rename(index={RR: "Rolls-Royce", MKT: "FTSE 100"})
risk_table.style.format({"Annualised return": "{:.1%}", "Annualised volatility": "{:.1%}", "Sharpe ratio": "{:.2f}"})

## 7. Maximum drawdown — Rolls-Royce and FTSE 100
Maximum drawdown is the largest peak-to-trough fall. It measures how much an investor could have lost, which volatility alone does not show.

In [ ]:
def drawdown_stats(series: pd.Series) -> dict:
    running_max = series.cummax()
    dd = series / running_max - 1
    trough = dd.idxmin()
    peak = series.loc[:trough].idxmax()
    recovered = series.loc[trough:][series.loc[trough:] >= series.loc[peak]]
    return {"Max drawdown": dd.min(), "Peak date": peak.date(), "Trough date": trough.date(),
            "Recovery date": recovered.index[0].date() if len(recovered) else "Not recovered",
            "Days peak→trough": (trough - peak).days, "series": dd}

dd = {name: drawdown_stats(prices[t]) for name, t in [("Rolls-Royce", RR), ("FTSE 100", MKT)]}
dd_table = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "series"} for k, v in dd.items()}).T
display(dd_table)

fig, ax = plt.subplots()
for name, v in dd.items():
    ax.plot(v["series"].index, v["series"], label=name)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(title="Drawdown from previous peak", ylabel="Drawdown")
ax.legend(); plt.show()

### Interpretation
Rolls-Royce's largest drawdown (about −55%, September 2021 to October 2022) was roughly four times deeper than the FTSE 100's over the same sample. The previous peak was only regained in March 2023, after the FY2022 results. Strong cumulative returns came with deep interim losses. The investment case has to account for that path risk, not only the end point.

## 8. Annual and monthly returns
Calendar-year returns from year-end prices. **2021 runs from 4 January (the first data point) and 2026 runs to 8 October**, so both are partial years and are flagged.

In [ ]:
year_end = prices.resample("YE").last()
annual = year_end.pct_change()
annual.iloc[0] = year_end.iloc[0] / prices.iloc[0] - 1      # 2021 from first observation
annual.index = annual.index.year
annual["RR minus FTSE (pp)"] = (annual[RR] - annual[MKT]) * 100
annual["Note"] = ""
annual.loc[annual.index.min(), "Note"] = "Partial: from 4 Jan"
annual.loc[annual.index.max(), "Note"] = f"Partial: YTD to {prices.index[-1]:%d %b}"
display(annual.rename(columns={RR: "Rolls-Royce", MKT: "FTSE 100"})
        .style.format({"Rolls-Royce": "{:.1%}", "FTSE 100": "{:.1%}", "RR minus FTSE (pp)": "{:+.1f}"}))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(annual)); w = 0.4
ax.bar(x - w/2, annual[RR] * 100, w, label="Rolls-Royce")
ax.bar(x + w/2, annual[MKT] * 100, w, label="FTSE 100")
ax.set_xticks(x, [f"{y}{'*' if n else ''}" for y, n in zip(annual.index, annual["Note"])])
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Calendar-year returns (* = partial year)", ylabel="Return (%)")
ax.legend(); plt.show()

In [ ]:
month_end = prices[RR].resample("ME").last()
monthly = month_end.pct_change().dropna()
last_month_partial = prices.index[-1] < month_end.index[-1] - pd.offsets.BDay(1)
if last_month_partial:
    print(f"Note: {month_end.index[-1]:%B %Y} is a partial month (data to {prices.index[-1]:%d %b}); excluded from rankings.")
    monthly = monthly.iloc[:-1]

ranked = pd.concat([monthly.nlargest(5).rename("Return").to_frame().assign(Direction="Best"),
                    monthly.nsmallest(5).rename("Return").to_frame().assign(Direction="Worst")])
ranked.index = ranked.index.strftime("%B %Y")
ranked.style.format({"Return": "{:+.1%}"})

## 9. Event windows around key announcements
Each window runs from **5 trading days before to 5 trading days after** the announcement (11 trading days), using positions in the trading calendar rather than calendar dates so every window has the same length. Relative return = Rolls-Royce cumulative return − FTSE 100 cumulative return over the same window.

Event dates are taken from the Rolls-Royce RNS / press-release archive (see sources in the report). The 12 April 2022 date marks a sharp sell-off referred to in the report as a broker downgrade; **verify the catalyst** before presenting it.

In [ ]:
EVENTS = pd.DataFrame([
    ("2022-04-12", "April 2022 sell-off (downgrade — verify)"),
    ("2022-08-04", "H1 2022 results"),
    ("2023-02-23", "FY2022 results"),
    ("2023-07-26", "July 2023 trading update"),
    ("2023-11-28", "Capital Markets Day"),
    ("2024-02-22", "FY2023 results"),
    ("2025-02-27", "FY2024 results"),
    ("2026-02-26", "FY2025 results"),
    ("2026-07-30", "H1 2026 results"),
], columns=["date", "event"])
EVENTS["date"] = pd.to_datetime(EVENTS["date"])

def event_window(date, pre=5, post=5):
    pos = returns.index.get_indexer([date])[0]
    if pos == -1:
        raise ValueError(f"{date.date()} is not a trading day in the data")
    return returns.iloc[pos - pre: pos + post + 1]

rows = []
for d, name in EVENTS.itertuples(index=False):
    win = event_window(d)
    cum = (1 + win).prod() - 1
    rows.append({"Event": name, "Date": d.date(),
                 "RR day return": returns.loc[d, RR], "FTSE day return": returns.loc[d, MKT],
                 "RR window": cum[RR], "FTSE window": cum[MKT], "Relative window": cum[RR] - cum[MKT]})
event_table = pd.DataFrame(rows).set_index("Event")
event_table.style.format({c: "{:+.1%}" for c in event_table.columns if c != "Date"})

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11), sharey=False)
for ax, (d, name) in zip(axes.flat, EVENTS.itertuples(index=False)):
    win = event_window(d)
    cum = ((1 + win).cumprod() - 1) * 100
    rel_days = np.arange(-5, 6)
    ax.plot(rel_days, cum[RR], marker="o", label="Rolls-Royce")
    ax.plot(rel_days, cum[MKT], marker="o", label="FTSE 100")
    ax.axvline(0, linestyle="--", color="grey")
    ax.set_title(f"{name}\n{d:%d %b %Y}", fontsize=10)
    ax.set_xlabel("Trading days relative to event")
axes.flat[0].set_ylabel("Cumulative return (%)"); axes.flat[0].legend(fontsize=8)
fig.suptitle("Cumulative returns around key events (−5 to +5 trading days)", y=1.0)
fig.tight_layout(); plt.show()

## 10. Market-model event study (abnormal returns and significance)
Raw relative returns assume a beta of 1. A **market model** estimates each event's normal relationship with the FTSE 100 from a pre-event estimation window and measures the surprise:

- Estimation window: trading days −250 to −30 before the event: $R_{RR,t} = \alpha + \beta R_{M,t} + \varepsilon_t$
- Abnormal return: $AR_t = R_{RR,t} - (\hat\alpha + \hat\beta R_{M,t})$
- Event-day test: $t = AR_0 / \hat\sigma_\varepsilon$; three-day test: $CAR_{[-1,+1]}$ with $t = CAR / (\hat\sigma_\varepsilon\sqrt{3})$
- Two-sided p-values from a t-distribution with $n-2$ degrees of freedom

These tests assume normally distributed, independent residuals. Overlapping news, volatility clustering and event selection all weaken that assumption, and statistical significance is not proof of causality.

In [ ]:
def p_value(t, dof):
    if HAVE_SCIPY:
        return 2 * stats.t.sf(abs(t), dof)
    return math.erfc(abs(t) / math.sqrt(2))            # normal approximation

def market_model_event(date, est_start=-250, est_end=-30):
    pos = returns.index.get_indexer([date])[0]
    est = returns.iloc[pos + est_start: pos + est_end]
    b, a = np.polyfit(est[MKT], est[RR], 1)
    resid = est[RR] - (a + b * est[MKT])
    sigma = resid.std(ddof=2)
    dof = len(est) - 2
    ar = returns[RR] - (a + b * returns[MKT])
    ar0 = ar.iloc[pos]
    car3 = ar.iloc[pos - 1: pos + 2].sum()
    t0, t3 = ar0 / sigma, car3 / (sigma * np.sqrt(3))
    return {"Alpha (daily)": a, "Beta": b, "Residual σ": sigma,
            "AR day 0": ar0, "t (day 0)": t0, "p (day 0)": p_value(t0, dof),
            "CAR [-1,+1]": car3, "t (CAR)": t3, "p (CAR)": p_value(t3, dof)}

mm = pd.DataFrame({name: market_model_event(d) for d, name in EVENTS.itertuples(index=False)}).T
mm["Significant at 5% (day 0)"] = np.where(mm["p (day 0)"] < 0.05, "Yes", "No")
mm.style.format({"Alpha (daily)": "{:.4f}", "Beta": "{:.2f}", "Residual σ": "{:.3%}",
                 "AR day 0": "{:+.1%}", "t (day 0)": "{:.2f}", "p (day 0)": "{:.4f}",
                 "CAR [-1,+1]": "{:+.1%}", "t (CAR)": "{:.2f}", "p (CAR)": "{:.4f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors = ["tab:green" if v > 0 else "tab:red" for v in mm["AR day 0"]]
ax.barh(mm.index, mm["AR day 0"] * 100, color=colors)
for i, (v, p) in enumerate(zip(mm["AR day 0"], mm["p (day 0)"])):
    ax.text(v * 100 + (0.5 if v > 0 else -0.5), i, f"p={p:.3f}", va="center", ha="left" if v > 0 else "right", fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
ax.set(title="Event-day abnormal returns (market model)", xlabel="Abnormal return (%)")
plt.show()

## 11. Summary statistics (exported for the report and Excel model)

In [ ]:
summary = pd.Series({
    "Period": f"{prices.index[0].date()} to {prices.index[-1].date()}",
    "RR total return": f"{total_return[RR]:.1%}",
    "FTSE total return": f"{total_return[MKT]:.1%}",
    "RR annualised return": f"{ann_return[RR]:.1%}",
    "FTSE annualised return": f"{ann_return[MKT]:.1%}",
    "RR annualised volatility": f"{ann_vol[RR]:.1%}",
    "FTSE annualised volatility": f"{ann_vol[MKT]:.1%}",
    "Correlation": f"{correlation:.2f}",
    "Raw beta": f"{beta:.2f}",
    "Blume-adjusted beta": f"{beta_blume:.2f}",
    f"RR Sharpe (rf {RISK_FREE:.1%})": f"{sharpe[RR]:.2f}",
    f"FTSE Sharpe (rf {RISK_FREE:.1%})": f"{sharpe[MKT]:.2f}",
    "RR max drawdown": f"{dd['Rolls-Royce']['Max drawdown']:.1%}",
    "FTSE max drawdown": f"{dd['FTSE 100']['Max drawdown']:.1%}",
    "Peak 30-day volatility (RR)": f"{rolling_vol[RR].max():.1%} on {peak_date.date()}",
}, name="Value")
summary.to_csv(OUT_DIR / "summary_statistics.csv")
event_table.to_csv(OUT_DIR / "event_windows.csv")
mm.to_csv(OUT_DIR / "market_model_event_study.csv")
annual.to_csv(OUT_DIR / "annual_returns.csv")
print(f"Saved CSVs to {OUT_DIR.resolve()}")
summary.to_frame()

## 12. Conclusions
1. **Exceptional, back-loaded returns.** Most of the gain came in 2023–2025, after the FY2022 results and the July 2023 trading update, the two largest single-day re-ratings in the sample.
2. **High, mostly stock-specific risk.** Volatility was more than three times the FTSE 100's and the maximum drawdown was more than half the share price. The correlation of about 0.5 means most of the risk was company-specific.
3. **Beta supports the CAPM input but is unstable.** The full-period beta (~1.65) is close to the model's 1.58. The rolling beta and the Blume adjustment show a real range, so the valuation is tested across discount rates of 8–13%.
4. **Results days carried the information.** The market-model event study shows large, statistically significant abnormal returns around the 2023 results and updates. Not every event was significant: the April 2022 sell-off was not significant at 5% (p ≈ 0.14). Compare the FY2025 and H1 2026 reactions with the 2023 ones: smaller abnormal returns on strong results would suggest the turnaround is now largely priced in.
5. **Link to valuation.** The share price has already rewarded the turnaround. The question for the investment thesis is whether future cash flows beat a much higher bar, and the reverse DCF in the Excel model quantifies that bar.

*Re-run the whole notebook (Kernel → Restart & Run All) before quoting numbers; check that conclusions 4 and 5 still match the tables above.*